# 02. 장기 작업 오케스트레이션과 재계획

## 목표
ER 계층이 하위 목표를 만들고 VLA 실행 결과를 관찰해 성공·실패를 판정하는 폐루프를 상태 기계로 구현합니다. 한 번 실패한 grasp를 재시도하는 이유도 살펴봅니다.

In [ ]:
from dataclasses import dataclass

@dataclass
class World:
    robot_at: str = 'start'
    holding: bool = False
    object_at: str = 'table'
    attempts: int = 0

def observe(world):
    return {'robot_at': world.robot_at, 'holding': world.holding, 'object_at': world.object_at}

def execute(action, world):
    # VLA와 실제 환경을 단순화한 결정론적 실행기입니다.
    if action.startswith('move:'):
        world.robot_at = action.split(':')[1]
        return True
    if action == 'grasp' and world.robot_at == world.object_at:
        world.attempts += 1
        world.holding = world.attempts >= 2  # 첫 시도 실패를 의도적으로 주입
        return world.holding
    if action == 'place' and world.robot_at == 'shelf' and world.holding:
        world.holding = False
        world.object_at = 'shelf'
        return True
    return False

In [ ]:
plan = ['move:table', 'grasp', 'move:shelf', 'place']
world, cursor, budget = World(), 0, 8

while cursor < len(plan) and budget > 0:
    action = plan[cursor]
    success = execute(action, world)
    print(f'{action:10} success={success} observation={observe(world)}')
    budget -= 1
    # 성공한 단계만 진행합니다. 실패한 grasp는 새 관찰을 받은 뒤 다시 시도합니다.
    if success:
        cursor += 1

goal_verified = world.object_at == 'shelf' and not world.holding
print('증거 기반 완료 판정:', goal_verified)

## 확장 과제

- grasp가 세 번 실패하면 다른 그리퍼를 요청하는 recovery branch를 추가하세요.
- 사람 근접 관찰이 들어오면 계획 cursor를 유지한 채 `safe_stop` 상태로 전환하세요.
- 종료 조건을 자연어 생성이 아니라 센서로 확인 가능한 predicate 조합으로 정의하세요.